In [3]:
import pandas as pd
import numpy as np
import seaborn as sn
import matplotlib.pyplot as plt

#rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

#sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict




# What is a QSPR Model?

**Quantitative Structure-Property Relationship (QSPR)** models use mathematical and machine learning methods to predict a property of a molecule from its chemical structure.

Because computers cannot directly interpret molecular structures, molecules are first converted into numerical values called **molecular descriptors**. These descriptors can represent properties such as molecular weight, lipophilicity (LogP), or polar surface area (TPSA).

In this example, we will develop a QSPR model that predicts the **corrosion inhibition efficiency (IE)** of organic molecules based on molecular descriptors calculated with **RDKit**.

**QSPR workflow:**

Molecule → Molecular Descriptors → Machine Learning Model → Predicted Inhibition Efficiency

The goal is to learn the relationship between molecular structure and inhibition efficiency, allowing us to estimate the performance of new molecules before experimental testing.

# Dataset

- The dataset consists of molecular structures and their corresponding **Inhibition Efficiency (IE)** values.
- Each **row** represents a single molecule.
- Each **column** contains a specific piece of information about that molecule.
- The **SMILES** column stores the molecular structure using the **Simplified Molecular Input Line Entry System (SMILES)**, a text-based notation for chemical structures.
- The **IE** column contains the inhibition efficiency value that we want to predict.
- In machine learning, the property being predicted is called the **target variable**.
- Additional molecular descriptors will later be calculated from the SMILES strings and used as **features** for model building.
- Features are the input variables used by the model, while the target variable is the output the model attempts to predict.
- Understanding the structure of the dataset is an important first step before exploring, preprocessing, and modeling the data.

In [ ]:
# read the dataset
#TODO: type in the name of the dataset from folder data_qspr
df = pd.read_csv('name_of_the_dataset.csv') 
df.head()

# Molecular Descriptors

- Molecular descriptors are numerical values that describe different properties of a molecule.
- They convert chemical structures into a format that can be used by machine learning algorithms.
- Descriptors can capture information about molecular size, shape, polarity, lipophilicity, and other chemical characteristics.
- In a QSPR model, descriptors serve as the **features** (input variables) used to predict a target property.
- Descriptors are calculated directly from the molecular structure using cheminformatics tools such as **RDKit**.

### Examples of Descriptors

- **Molecular Weight (MW):** The total mass of all atoms in a molecule.
- **LogP:** A measure of lipophilicity, indicating how readily a molecule partitions between water and a nonpolar solvent.
- **Topological Polar Surface Area (TPSA):** An indicator of molecular polarity and the ability to form interactions through polar atoms.
- **Number of Hydrogen Bond Donors (HBD):** The number of atoms capable of donating hydrogen bonds.

- Different descriptors provide different information about a molecule, and together they help the model learn relationships between molecular structure and inhibition efficiency.
- More infos on how to get started: https://rdkit.readthedocs.io/en/latest/GettingStartedInPython.html

In [ ]:
# find the SMILES of your favourite molecule and put it in
m = Chem.MolFromSmiles('SMILES_Of_your_favourite_molecule')
m

In [ ]:
Descriptors.MolWt(m)

In [ ]:
#TODO 
# calculate LogP and TPSA for your favourite molecule

In [ ]:
# Convert SMILES to RDKit Mol objects
df["Mol"] = df["SMILES"].apply(Chem.MolFromSmiles)

# Calculate descriptors
df["MolWt"] = df["Mol"].apply(Descriptors.MolWt)
df["MolLogP"] = df["Mol"].apply(Descriptors.MolLogP)
df["TPSA"] = df["Mol"].apply(Descriptors.TPSA)

# Optional: remove the Mol column
df = df.drop(columns=["Mol"])

df.head()

# Data exploration

- Before building a model, it is important to understand the data and identify potential issues.
- Data exploration helps us understand the characteristics of both the target variable and the molecular descriptors.
- Visualizing the data can reveal patterns, trends, and unusual observations that may affect model performance.
- Histograms can be used to examine the distribution of inhibition efficiency (IE) and molecular descriptors.
- An uneven distribution of the target variable may cause the model to perform better for some value ranges than others.
- Summary statistics such as the mean, median, minimum, and maximum provide a quick overview of the dataset.
- Boxplots can help identify potential outliers, which are observations that differ substantially from the majority of the data.
- Correlation analysis can reveal relationships between descriptors and help identify highly correlated features.
- Understanding the data is a critical step before preprocessing and model development.

### Questions to Consider

- Is the target variable evenly distributed across its range?
- Are there any unusually large or small values?
- Which descriptors show the greatest variation?
- Are some descriptors strongly correlated with one another?
- Are there patterns that might influence model performance?

In [ ]:
# Display the first few rows
df.head()

In [ ]:
# Number of rows and columns
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

In [ ]:
# Column names
df.columns

In [ ]:
# TODO:
# Display the last 5 rows of the dataset

# Your code here

In [ ]:
df["IE"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["IE"], bins=30, edgecolor="black")

plt.xlabel("Inhibition Efficiency (IE)")
plt.ylabel("Count")
plt.title("Distribution of IE")

plt.show()

In [ ]:
# Boxplot:
# median
# middle 50% of the data
# lower (25%) and upper (75%) quantile
# Whiskers (extend to the most extreme values that are not considered outliers)
# Outliers
plt.figure(figsize=(8,5))

plt.boxplot(df["IE"])

plt.ylabel("IE")
plt.title("Boxplot of Inhibition Efficiency")

plt.show()

In [ ]:
df.nlargest(5, "IE")

In [ ]:
df.nsmallest(5, "IE")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

axes[0].hist(df["MolWt"], bins=30)
axes[0].set_title("MolWt")

axes[1].hist(df["MolLogP"], bins=30)
axes[1].set_title("MolLogP")

axes[2].hist(df["TPSA"], bins=30)
axes[2].set_title("TPSA")

plt.tight_layout()
plt.show()

In [ ]:
# TODO:
# Calculate summary statistics for one feature column

# Your code here

In [ ]:
# TODO:
# Create a boxplot for Molecular Weight (MolWt)

# Your code here


In [ ]:
corr = df[["MolWt", "MolLogP", "TPSA", "IE"]].corr()

corr

In [ ]:
plt.figure(figsize=(6,5))

sn.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(df["MolWt"], df["IE"], alpha=0.6)

plt.xlabel("MolWt")
plt.ylabel("IE")

plt.title("MolWt vs IE")

plt.show()

In [ ]:
#TODO
# create the same kind of scatterplot for MolLogP and TPSA

# Your code here


In [ ]:
# Challenge:
#
# 1. Calculate one additional RDKit descriptor.
# 2. Add it to the dataframe.
# 3. Plot its distribution.
# 4. Check its correlation with IE.

# Data Preprocessing

- Data preprocessing prepares the dataset for machine learning.
- Raw data often contains issues that can negatively affect model performance.
- Common preprocessing steps include handling missing values, removing unnecessary features, and scaling numerical data.
- Missing values can prevent some machine learning algorithms from working correctly.
- Features with very little variation may provide little useful information and can sometimes be removed.
- Molecular descriptors can have very different ranges (e.g., Molecular Weight vs. LogP).
- Feature scaling transforms descriptors to a similar scale, which can improve the performance of many machine learning algorithms.
- Preprocessing should be performed using only the training data to avoid introducing information from the test set.
- A consistent preprocessing workflow helps create more reliable and reproducible models.

### Common Preprocessing Steps

1. Check for missing values.
2. Remove or handle invalid entries.
3. Examine descriptor ranges.
4. Scale numerical features if required.
5. Create training and test datasets.

In [ ]:
# Check for Missing Values
df.isnull().sum()

In [ ]:
df.isnull().sum().sum()

In [ ]:
df["Constant_Feature"] = 1


In [ ]:
# calculate variance:
# variance is a measure of how much a feature's values are spread out around the mean
df[["MolWt", "MolLogP", "TPSA", "Constant_Feature"]].var()

In [ ]:
#remove feature with low/no variance:
df.drop('Constant_Feature', axis=1, inplace=True)
df.head()

In [ ]:
#correlation:

np.random.seed(42)
df["MolWt_Copy"] = df["MolWt"] * 1.05 + np.random.normal(0, 5, len(df))
corr = df[["MolWt", "MolWt_Copy", "MolLogP", "TPSA", "IE"]].corr()

plt.figure(figsize=(7,6))

sn.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
#TODO:
# drop the highly correlated feature MolWt_Copy

#your code here


In [ ]:
# feature and target separation
X = df[["MolWt", "MolLogP", "TPSA"]]

y = df["IE"]

print(X.head())
print(y.head())

## Train-Test Split

- Machine learning models must be evaluated on data they have not seen during training.
- The dataset is therefore divided into a **training set** and a **test set**.
- The **training set** is used to learn relationships between descriptors and the target property.
- The **test set** is used to assess how well the model performs on new, unseen molecules.
- A common split is **80% training data** and **20% test data**.
- Keeping the test set separate helps detect **overfitting**, where a model performs well on training data but poorly on new data.
- The test set should only be used for final model evaluation and not during model training.

In [ ]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

In [ ]:
#scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Convert back to a DataFrame for inspection:
pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns
).head()

In [ ]:
X_train.describe()

In [ ]:
pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns
).describe()

# Building the First QSPR Model

- We will use a **Linear Regression** model to predict inhibition efficiency (IE) from molecular descriptors.
- During training, the model learns the relationship between the descriptors and the target variable.
- Once trained, the model can predict IE values for molecules it has not seen before.
- To evaluate the model, predictions are compared to the experimental values in the test set.
- The quality of the model can be assessed using statistical metrics such as **R²**, **MAE**, and **RMSE**.
- A good model should make accurate predictions not only for the training data, but also for new, unseen molecules.

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

In [ ]:
# Predictions for training and test sets

y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

In [ ]:
results = pd.DataFrame({
    "Experimental": y_test,
    "Predicted": y_test_pred
})

results.head(10)

In [ ]:
# Training metrics
train_r2 = r2_score(y_train, y_train_pred)
train_rmse = root_mean_squared_error(y_train, y_train_pred)

# Test metrics
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)

print("Training Set")
print(f"R²   : {train_r2:.3f}")
print(f"RMSE : {train_rmse:.2f}")

print("\nTest Set")
print(f"R²   : {test_r2:.3f}")
print(f"RMSE : {test_rmse:.2f}")

In [ ]:
plt.figure(figsize=(7,7))

# Training data
plt.scatter(
    y_train,
    y_train_pred,
    alpha=0.5,
    label="Training Set"
)

# Test data
plt.scatter(
    y_test,
    y_test_pred,
    alpha=0.8,
    label="Test Set"
)

# Perfect prediction line
min_val = min(y.min(), y_train_pred.min(), y_test_pred.min())
max_val = max(y.max(), y_train_pred.max(), y_test_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--",
    label="Perfect Prediction"
)

plt.xlabel("Experimental IE")
plt.ylabel("Predicted IE")

plt.title("Predicted vs Experimental Values")

plt.legend()

plt.show()

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

y_train_pred_rf = rf.predict(X_train)
y_test_pred_rf = rf.predict(X_test)

In [ ]:
print("Training Set")
print(f"R²   : {r2_score(y_train, y_train_pred_rf):.3f}")
print(f"RMSE : {root_mean_squared_error(y_train, y_train_pred_rf):.2f}")

print("\nTest Set")
print(f"R²   : {r2_score(y_test, y_test_pred_rf):.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_test_pred_rf):.2f}")

In [ ]:
plt.figure(figsize=(7,7))

plt.scatter(
    y_train,
    y_train_pred_rf,
    alpha=0.5,
    label="Training Set"
)

plt.scatter(
    y_test,
    y_test_pred_rf,
    alpha=0.8,
    label="Test Set"
)

plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    "r--",
    label="Perfect Prediction"
)

plt.xlabel("Experimental IE")
plt.ylabel("Predicted IE")

plt.title("Random Forest Predictions")

plt.legend()

plt.show()

In [ ]:
# Challenge:
#
# Train a Linear Regression model
# using only MolWt and MolLogP.
#
# Compare the performance metrics
# to the model using all descriptors.

In [ ]:
# Add random descriptors for demostration purpose
for i in range(10):
    X[f"Random_{i}"] = np.random.normal(size=len(X))

# Smaller training set
X_train_special, X_test_special, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.7,
    random_state=42
)

decision_tree_model = DecisionTreeRegressor(
    max_depth=None,
    random_state=42
)

decision_tree_model.fit(X_train_special, y_train)

y_train_pred = decision_tree_model.predict(X_train_special)
y_test_pred = decision_tree_model.predict(X_test_special)

In [ ]:
print("Train R²:", r2_score(y_train, y_train_pred))
print("Test R² :", r2_score(y_test, y_test_pred))

In [ ]:
plt.figure(figsize=(7,7))

# Training data
plt.scatter(
    y_train,
    y_train_pred,
    alpha=0.5,
    label="Training Set"
)

# Test data
plt.scatter(
    y_test,
    y_test_pred,
    alpha=0.8,
    label="Test Set"
)

# Perfect prediction line
min_val = min(y.min(), y_train_pred.min(), y_test_pred.min())
max_val = max(y.max(), y_train_pred.max(), y_test_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--",
    label="Perfect Prediction"
)

plt.xlabel("Experimental IE")
plt.ylabel("Predicted IE")

plt.title("Predicted vs Experimental Values")

plt.legend()

plt.show()

# Model Performance: Underfitting vs. Overfitting

- A good machine learning model should perform well on both the training data and unseen test data.
- Comparing training and test errors helps us assess whether a model is learning the underlying patterns correctly.

### Underfitting

- The model is too simple to capture the relationship between the descriptors and the target property.
- Training error is high.
- Test error is high.
- The model performs poorly on both known and new data.

### Good Fit

- The model captures the important patterns in the data without memorizing individual samples.
- Training error is low.
- Test error is low.
- The model generalizes well to new molecules.

### Overfitting

- The model learns not only the underlying patterns but also random noise in the training data.
- Training error is very low.
- Test error is significantly higher than the training error.
- Predictions for new molecules become less reliable.

### Rule of Thumb

- **High training error + high test error → Underfitting**
- **Low training error + low test error → Good fit**
- **Very low training error + much higher test error → Overfitting**

# Cross-Validation

- A single train-test split provides only one estimate of model performance.
- Results may vary depending on which samples happen to be assigned to the training and test sets.
- Cross-validation evaluates a model multiple times using different subsets of the data.
- In **k-fold cross-validation**, the dataset is divided into *k* groups (folds).
- The model is trained on *k-1* folds and tested on the remaining fold.
- This process is repeated until every fold has been used as the test set exactly once.
- The final score is the average performance across all folds.
- Cross-validation provides a more reliable estimate of how well a model generalizes to new molecules.
- A large variation between folds may indicate that the model is sensitive to the training data.


In [ ]:
model = RandomForestRegressor()

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² Scores for each fold:")
print(cv_scores)

print(f"\nMean R²: {cv_scores.mean():.3f}")
print(f"Standard Deviation: {cv_scores.std():.3f}")

In [ ]:
plt.figure(figsize=(7,4))

plt.bar(
    range(1, len(cv_scores)+1),
    cv_scores
)

plt.xlabel("Fold")
plt.ylabel("R²")
plt.title("5-Fold Cross-Validation Scores")

plt.ylim(0, 1)

plt.show()

In [ ]:
y_cv_pred = cross_val_predict(
    model,
    X,
    y,
    cv=5
)

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(
    y,
    y_cv_pred,
    alpha=0.7
)

plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    "r--",
    label="Perfect Prediction"
)

plt.xlabel("Experimental IE")
plt.ylabel("Predicted IE")
plt.title("Cross-Validated Predictions")

plt.legend()

plt.show()

In [ ]:
# Challenge:
# Repeat the cross-validation using 10 folds instead of 5.
#
# Compare the mean R² value to the 5-fold result.

# Your code here

# Feature Importance

- Feature importance helps us understand which descriptors have the greatest influence on model predictions.
- Identifying important descriptors can provide insight into the relationship between molecular structure and the target property.
- Some machine learning models, such as Random Forests, can directly estimate feature importance.
- Features with higher importance contribute more strongly to the model's predictions.
- Features with very low importance may contribute little useful information.
- Feature importance can help guide descriptor selection and simplify models.
- It is important to remember that feature importance indicates the influence on predictions and does not necessarily imply a causal relationship.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

In [ ]:
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance


In [ ]:
plt.figure(figsize=(7,4))

plt.bar(
    feature_importance["Feature"],
    feature_importance["Importance"]
)

plt.xlabel("Descriptor")
plt.ylabel("Importance")
plt.title("Feature Importance")

plt.show()

In [ ]:
# Challenge:
#
# Remove the most important descriptor.
# Retrain the Random Forest model.
# Compare the new R² value to the original model.
#
# What happens to model performance?

# Applicability Domain and Limitations

## Applicability Domain

- A QSPR model is most reliable for molecules that are similar to those used during training.
- The range of chemical structures represented in the training data defines the model's **applicability domain**.
- Predictions made for molecules outside this domain should be treated with caution.
- For example, a model trained on small organic molecules may not perform well for very large or structurally different compounds.
- Before trusting a prediction, it is important to consider whether the new molecule is sufficiently similar to the training set.
- A prediction can always be generated, but not every prediction is equally reliable.

## Limitations of QSPR Models

- QSPR models can only learn patterns that are present in the data used for training.
- Model quality depends strongly on the quality and diversity of the available data.
- Experimental errors and uncertainties in measured properties can affect model performance.
- Important molecular characteristics may not be captured by the selected descriptors.
- A good prediction does not necessarily provide an explanation of the underlying chemical mechanism.
- Models may perform differently when applied to new datasets or chemical spaces.
- Correlation does not imply causation: a descriptor may be associated with a property without directly causing it.
- Machine learning models should be considered decision-support tools rather than replacements for experimental validation.

## Key Takeaways

- Reliable predictions require both a well-trained model and appropriate input data.
- Predictions are most trustworthy within the model's applicability domain.
- Understanding model limitations is essential for the responsible use of QSPR models.
- Experimental validation remains an important step when evaluating new corrosion inhibitors.